# FLUX.3-Image-Klein 免 VAE 直出像素头（Direct Pixel Head）全云端一键训练与评测台（Colab L4 24GB 专用）

本 Notebook 全程运行于 **Google Colab L4 (24GB VRAM)**，**本机零算力负担**，一键完成 **`FLUX.3-Image-Klein` Direct RGB Pixel Head** 的校准集构建、离线潜空间预编码、**FlowMap 蒸馏 + Base 双分支交替训练**与三方跑分对比：

- **核心原理（`FastFrames` `arXiv:2611.23226`）**：冻结 `FLUX.3-Image-Klein` DiT 主干，在 `final_layer["image"].linear` 前置挂钩截获已剥离参考图与文本的 **`3072` 维图像特征**与去噪后 **`128` 通道潜变量 `x0_hat_norm`**，经轻量 `ConvNeXt + PixelShuffle` 头（`~24.4M` 参数）在 **~4.5 ms** 内直出高清 RGB。
- **单权重通吃全生态**：
  1. 默认同时交替抽样 **`flux3-image-klein`（FlowMap 4~16 步，$\sigma \in [0.28, 0.72]$，自动注入 `sample_sigmas=[sigma, 0.0]`）** 与 **`flux3-image-klein-base`（25~50 步 CFG，$\sigma \in [0.10, 0.38]$）**；
  2. 天然兼容 **T2I 文生图、I2I 图生图、MI2I 多参考图指令编辑**；
  3. 评测阶段自动同台对比 **官方原版 `flux2-vae.safetensors`（Teacher）**、**BFL 官方轻量 `full_encoder_small_decoder.safetensors`** 与 **你的 `Flux3PixelHead`**。

### 一键运行顺序
确认右上角运行时为 **`Python 3` + `L4 GPU`** ➔ 点击 **「代码执行程序 (Runtime) ➔ 全部运行 (Run all)」**（单元格 6 为可选 `2×` 超分直出头加训）。

In [ ]:
# @title 1. 挂载 Google Drive、校验 L4 显卡并自动部署 Clean-Room MIT 插件与对比工作流
import subprocess
from pathlib import Path
from google.colab import drive

subprocess.run(["nvidia-smi"], check=False)
drive.mount("/content/drive")

DRIVE_HUB = Path("/content/drive/MyDrive/ComfyUI_Hub")
PROJECT_DRIVE = Path("/content/drive/MyDrive/FastFrames_Flux3Klein")
CKPT_DRIVE = PROJECT_DRIVE / "checkpoints"
EVAL_DRIVE = PROJECT_DRIVE / "eval_report"
PIXEL_HEADS_DRIVE = DRIVE_HUB / "models/pixel_heads"
VAE_DRIVE = DRIVE_HUB / "models/vae"
CUSTOM_NODE_DRIVE = DRIVE_HUB / "custom_nodes_flux3/ComfyUI-Flux3-PixelHead"
WF_DRIVE_DIR = DRIVE_HUB / "user_flux3/default/workflows"
LOCAL_SRC = Path("/content/ComfyUI-Flux3-PixelHead")

for d in (CKPT_DRIVE, EVAL_DRIVE, PIXEL_HEADS_DRIVE, VAE_DRIVE, CUSTOM_NODE_DRIVE, WF_DRIVE_DIR, LOCAL_SRC):
    d.mkdir(parents=True, exist_ok=True)

FILES = {'__init__.py': 'from .nodes import NODE_CLASS_MAPPINGS, NODE_DISPLAY_NAME_MAPPINGS\n\n__all__ = ["NODE_CLASS_MAPPINGS", "NODE_DISPLAY_NAME_MAPPINGS"]\n', 'flux3_pixel_head.py': '"""Clean-room PyTorch implementation of Direct Pixel Readout Head for FLUX.3-Image-Klein.\n\nLicense: MIT\nArchitecture follows the public mathematical formulation of FastFrames (arXiv:2611.23226):\n  - Frozen FLUX.3-Image-Klein DiT (`comfy.ldm.flux3.model.Flux3`, hidden_size=3072, in_channels=128, 16x spatial).\n  - Pre-hook taps `f` (`[B, H_lat*W_lat, 3072]`, post-`modulated_norm`, pre-`final_layer["image"].linear`)\n    plus the exact denoised latent `x0_hat_norm` (`[B, 128, H_lat, W_lat]`, sigma-invariant across FlowMap & Base).\n  - Feature LayerNorm + bridge linear (`3072 -> width`, bias=False) + latent linear (`128 -> width`, bias=True)\n    + entry LayerNorm at 16px token grid.\n  - `depth_low` ConvNeXt2d blocks at 16px token grid -> 2x PixelShuffle -> `depth_high` ConvNeXt2d blocks at 8px grid.\n  - Final LayerNorm + FP32 Linear (`width//2 -> 8*8*out_channels`) producing RGB in `[-1, 1]`.\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Optional\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nTOKEN_PIXELS = 16  # Flux2/Flux3 VAE spatial downscale is 16x, DiT patch is 1x1\n\n\nclass ConvNeXtBlock2d(nn.Module):\n    """2D ConvNeXt block with 3x3 depthwise conv, channel LayerNorm, 4x GELU MLP, and LayerScale."""\n\n    def __init__(self, dim: int, mlp_ratio: int = 4, ls_init: float = 1.0):\n        super().__init__()\n        self.dw = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim)\n        self.norm = nn.LayerNorm(dim, eps=1e-6)\n        hidden = dim * mlp_ratio\n        self.fc1 = nn.Linear(dim, hidden)\n        self.fc2 = nn.Linear(hidden, dim)\n        self.gamma = nn.Parameter(torch.full((dim,), float(ls_init)))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        h = self.dw(x).movedim(1, -1)\n        h = self.fc2(F.gelu(self.fc1(self.norm(h))))\n        h = (h * self.gamma).movedim(-1, 1)\n        return x + h\n\n\ndef unpatchify_2d(tokens: torch.Tensor, patch: int, out_channels: int) -> torch.Tensor:\n    """Convert `[B, gh, gw, patch * patch * C]` -> `[B, C, gh * patch, gw * patch]`."""\n    b, gh, gw, _ = tokens.shape\n    x = tokens.view(b, gh, gw, patch, patch, out_channels)\n    x = x.permute(0, 5, 1, 3, 2, 4).contiguous()\n    return x.view(b, out_channels, gh * patch, gw * patch)\n\n\nclass Flux3PixelHead(nn.Module):\n    """1x Direct Pixel Head for FLUX.3-Image-Klein (`16x16` RGB pixels per DiT token)."""\n\n    def __init__(\n        self,\n        d_feat: int = 3072,\n        d_lat: int = 128,\n        width: int = 1024,\n        depth_low: int = 2,\n        depth_high: int = 1,\n        out_channels: int = 3,\n        ls_init: float = 1.0,\n    ):\n        super().__init__()\n        if width % 2 != 0:\n            raise ValueError(f"width must be even, got {width}")\n        self.d_feat = int(d_feat)\n        self.d_lat = int(d_lat)\n        self.width = int(width)\n        self.depth_low = int(depth_low)\n        self.depth_high = int(depth_high)\n        self.out_channels = int(out_channels)\n        self.ls_init = float(ls_init)\n\n        w_high = width // 2\n        self.out_patch = TOKEN_PIXELS // 2  # 8x8 pixels per 2x-upsampled position\n\n        self.feat_norm = nn.LayerNorm(self.d_feat, eps=1e-6)\n        self.feat_proj = nn.Linear(self.d_feat, width, bias=False)\n        self.lat_proj = nn.Linear(self.d_lat, width, bias=True)\n        self.entry_norm = nn.LayerNorm(width, eps=1e-6)\n\n        self.low_blocks = nn.ModuleList(\n            [ConvNeXtBlock2d(width, ls_init=ls_init) for _ in range(depth_low)]\n        )\n        self.up_norm = nn.LayerNorm(width, eps=1e-6)\n        self.up_proj = nn.Linear(width, 4 * w_high)\n        self.high_blocks = nn.ModuleList(\n            [ConvNeXtBlock2d(w_high, ls_init=ls_init) for _ in range(depth_high)]\n        )\n        self.norm_out = nn.LayerNorm(w_high, eps=1e-6)\n        self.out_proj = nn.Linear(w_high, self.out_patch * self.out_patch * self.out_channels)\n        nn.init.normal_(self.out_proj.weight, mean=0.0, std=0.02)\n        nn.init.zeros_(self.out_proj.bias)\n\n        self.meta: dict[str, str] = {}\n\n    def forward_trunk(self, feat: torch.Tensor, x0_hat_norm: torch.Tensor) -> torch.Tensor:\n        """Run shared trunk up to `high_blocks` (`[B, width//2, 2*gh, 2*gw]`)."""\n        b, _, gh, gw = x0_hat_norm.shape\n        if feat.ndim == 3:\n            feat = feat.view(b, gh, gw, self.d_feat)\n        lat_tok = x0_hat_norm.movedim(1, -1)  # [B, gh, gw, d_lat]\n\n        h = self.entry_norm(self.feat_proj(self.feat_norm(feat)) + self.lat_proj(lat_tok))\n        h = h.movedim(-1, 1).contiguous()\n        for blk in self.low_blocks:\n            h = blk(h)\n\n        u = self.up_proj(self.up_norm(h.movedim(1, -1))).movedim(-1, 1).contiguous()\n        h = F.pixel_shuffle(u, 2)\n        for blk in self.high_blocks:\n            h = blk(h)\n        return h\n\n    def forward(self, feat: torch.Tensor, x0_hat_norm: torch.Tensor) -> torch.Tensor:\n        """Return `[B, out_channels, gh*16, gw*16]` in `[-1, 1]`."""\n        h = self.forward_trunk(feat, x0_hat_norm)\n        h_tok = self.norm_out(h.movedim(1, -1))\n        with torch.autocast(device_type=h_tok.device.type, enabled=False):\n            patches = self.out_proj(h_tok.float())\n        return unpatchify_2d(patches, self.out_patch, self.out_channels)\n\n\nclass Flux3PixelHeadX2(nn.Module):\n    """Optional 2x Super-Resolution Direct Pixel Head (`32x32` pixels per DiT token)."""\n\n    def __init__(\n        self,\n        base: Flux3PixelHead,\n        depth_x2: int = 2,\n        ls_init: float = 1.0,\n    ):\n        super().__init__()\n        self.base = base\n        self.depth_x2 = int(depth_x2)\n        w_in = base.width // 2\n        w_x2 = w_in // 2\n        self.out_channels = base.out_channels\n        self.out_patch = base.out_patch\n\n        self.up2_norm = nn.LayerNorm(w_in, eps=1e-6)\n        self.up2_proj = nn.Linear(w_in, 4 * w_x2)\n        self.x2_blocks = nn.ModuleList(\n            [ConvNeXtBlock2d(w_x2, ls_init=ls_init) for _ in range(depth_x2)]\n        )\n        self.norm2 = nn.LayerNorm(w_x2, eps=1e-6)\n        self.out2_proj = nn.Linear(w_x2, self.out_patch * self.out_patch * self.out_channels)\n        nn.init.normal_(self.out2_proj.weight, mean=0.0, std=0.02)\n        nn.init.zeros_(self.out2_proj.bias)\n        self.meta: dict[str, str] = {}\n\n    def forward(self, feat: torch.Tensor, x0_hat_norm: torch.Tensor, scale: int = 2) -> torch.Tensor:\n        h = self.base.forward_trunk(feat, x0_hat_norm)\n        if scale == 1:\n            h1 = self.base.norm_out(h.movedim(1, -1))\n            with torch.autocast(device_type=h1.device.type, enabled=False):\n                p1 = self.base.out_proj(h1.float())\n            return unpatchify_2d(p1, self.base.out_patch, self.out_channels)\n\n        u2 = self.up2_proj(self.up2_norm(h.movedim(1, -1))).movedim(-1, 1).contiguous()\n        g = F.pixel_shuffle(u2, 2)\n        for blk in self.x2_blocks:\n            g = blk(g)\n        h2 = self.norm2(g.movedim(1, -1))\n        with torch.autocast(device_type=h2.device.type, enabled=False):\n            p2 = self.out2_proj(h2.float())\n        return unpatchify_2d(p2, self.out_patch, self.out_channels)\n\n\ndef save_flux3_pixel_head(\n    head: Flux3PixelHead | Flux3PixelHeadX2,\n    path: str | Path,\n    extra_meta: Optional[dict[str, str]] = None,\n) -> None:\n    from safetensors.torch import save_file\n\n    is_x2 = isinstance(head, Flux3PixelHeadX2)\n    base = head.base if is_x2 else head\n    meta = {\n        "arch": "flux3_pixel_head_x2" if is_x2 else "flux3_pixel_head",\n        "d_feat": str(base.d_feat),\n        "d_lat": str(base.d_lat),\n        "width": str(base.width),\n        "depth_low": str(base.depth_low),\n        "depth_high": str(base.depth_high),\n        "out_channels": str(base.out_channels),\n        "ls_init": str(base.ls_init),\n        "scale": "2" if is_x2 else "1",\n        "latent_input": "x0_hat_norm",\n    }\n    if is_x2:\n        meta["depth_x2"] = str(head.depth_x2)\n    if extra_meta:\n        for k, v in extra_meta.items():\n            meta[str(k)] = str(v)\n\n    sd = {k: v.detach().cpu().contiguous() for k, v in head.state_dict().items()}\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    save_file(sd, str(path), metadata=meta)\n\n\ndef load_flux3_pixel_head(\n    path: str | Path,\n    device: str | torch.device = "cpu",\n    dtype: torch.dtype = torch.float32,\n) -> Flux3PixelHead | Flux3PixelHeadX2:\n    from safetensors import safe_open\n    from safetensors.torch import load_file\n\n    path = Path(path)\n    with safe_open(str(path), framework="pt", device="cpu") as fh:\n        meta = dict(fh.metadata() or {})\n\n    arch = meta.get("arch", "flux3_pixel_head")\n    base = Flux3PixelHead(\n        d_feat=int(meta.get("d_feat", "3072")),\n        d_lat=int(meta.get("d_lat", "128")),\n        width=int(meta.get("width", "1024")),\n        depth_low=int(meta.get("depth_low", "2")),\n        depth_high=int(meta.get("depth_high", "1")),\n        out_channels=int(meta.get("out_channels", "3")),\n        ls_init=float(meta.get("ls_init", "1.0")),\n    )\n    if arch == "flux3_pixel_head_x2" or "depth_x2" in meta:\n        model: nn.Module = Flux3PixelHeadX2(\n            base=base,\n            depth_x2=int(meta.get("depth_x2", "2")),\n            ls_init=float(meta.get("ls_init", "1.0")),\n        )\n    else:\n        model = base\n\n    sd = load_file(str(path), device="cpu")\n    model.load_state_dict(sd, strict=True)\n    model = model.to(device=device, dtype=dtype)\n    if isinstance(model, Flux3PixelHeadX2):\n        model.base.out_proj.float()\n        model.out2_proj.float()\n    else:\n        model.out_proj.float()\n    model.meta = meta\n    model.eval()\n    return model\n', 'LICENSE': 'MIT License\n\nCopyright (c) 2026\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'nodes.py': '"""ComfyUI Nodes for FLUX.3-Image-Klein Direct Pixel Head (Zero-VAE RGB Decoding).\n\nLicense: MIT\n"""\n\nfrom __future__ import annotations\n\nimport os\nimport time\nimport logging\n\nimport folder_paths\nimport comfy.model_management\nimport torch\n\nfrom .flux3_pixel_head import (\n    Flux3PixelHeadX2,\n    load_flux3_pixel_head,\n)\nfrom .tap import attach_flux3_tap\n\nlogger = logging.getLogger(__name__)\n\nPIXEL_HEAD_DIR = os.path.join(folder_paths.models_dir, "pixel_heads")\nos.makedirs(PIXEL_HEAD_DIR, exist_ok=True)\nif "pixel_heads" not in folder_paths.folder_names_and_paths:\n    folder_paths.add_model_folder_path("pixel_heads", PIXEL_HEAD_DIR)\n\n\ndef _list_candidate_files() -> list[str]:\n    seen = set()\n    items = []\n    for folder_key in ("pixel_heads", "vae"):\n        if folder_key in folder_paths.folder_names_and_paths:\n            for fn in folder_paths.get_filename_list(folder_key):\n                if fn.endswith(".safetensors") and fn not in seen:\n                    seen.add(fn)\n                    items.append(fn)\n    return sorted(items) if items else ["flux3_klein_pixel_head.safetensors"]\n\n\ndef _resolve_head_path(name: str) -> str:\n    for folder_key in ("pixel_heads", "vae"):\n        if folder_key in folder_paths.folder_names_and_paths:\n            p = folder_paths.get_full_path(folder_key, name)\n            if p and os.path.exists(p):\n                return p\n    raise FileNotFoundError(\n        f"未找到 Pixel Head 权重 `{name}`，请将其放入 `ComfyUI/models/pixel_heads/` 或 `ComfyUI/models/vae/`。"\n    )\n\n\nclass Flux3PixelHeadLoader:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\n            "required": {\n                "head_name": (_list_candidate_files(),),\n                "precision": (["bf16_trunk_fp32_out", "fp32"], {"default": "bf16_trunk_fp32_out"}),\n            }\n        }\n\n    RETURN_TYPES = ("FLUX3_PIXEL_HEAD",)\n    RETURN_NAMES = ("pixel_head",)\n    FUNCTION = "load_head"\n    CATEGORY = "advanced/flux3"\n    TITLE = "Load FLUX.3 Klein Pixel Head"\n\n    def load_head(self, head_name: str, precision: str):\n        path = _resolve_head_path(head_name)\n        dtype = torch.bfloat16 if precision == "bf16_trunk_fp32_out" else torch.float32\n        head = load_flux3_pixel_head(path, device="cpu", dtype=dtype)\n        return (head,)\n\n\nclass Flux3PixelHeadTap:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\n            "required": {\n                "model": ("MODEL",),\n            }\n        }\n\n    RETURN_TYPES = ("MODEL", "FLUX3_TAP")\n    RETURN_NAMES = ("model", "pixel_tap")\n    FUNCTION = "install_tap"\n    CATEGORY = "advanced/flux3"\n    TITLE = "FLUX.3 Klein Pixel Head Tap"\n\n    def install_tap(self, model):\n        patched_model, tap_state = attach_flux3_tap(model)\n        return (patched_model, tap_state)\n\n\nclass Flux3PixelDecode:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\n            "required": {\n                "samples": ("LATENT",),\n                "pixel_head": ("FLUX3_PIXEL_HEAD",),\n                "pixel_tap": ("FLUX3_TAP",),\n                "output_scale": (["auto", "1x", "2x"], {"default": "auto"}),\n            }\n        }\n\n    RETURN_TYPES = ("IMAGE",)\n    RETURN_NAMES = ("image",)\n    FUNCTION = "decode"\n    CATEGORY = "advanced/flux3"\n    TITLE = "FLUX.3 Klein Pixel Decode (Zero VAE)"\n\n    @torch.no_grad()\n    def decode(self, samples, pixel_head, pixel_tap, output_scale: str = "auto"):\n        _ = samples["samples"]\n        f_cpu, xt_cpu, sigma = pixel_tap.resolve_final_features()\n\n        device = comfy.model_management.get_torch_device()\n        is_x2 = isinstance(pixel_head, Flux3PixelHeadX2)\n        if output_scale == "auto":\n            scale = 2 if is_x2 else 1\n        elif output_scale == "2x":\n            if not is_x2:\n                raise ValueError("当前加载的是 1x Pixel Head 权重，若需 2x 直出请加载带 `_x2` 的二阶权重。")\n            scale = 2\n        else:\n            scale = 1\n\n        head_dev = pixel_head.to(device)\n        trunk_dtype = next(head_dev.parameters()).dtype\n        f_dev = f_cpu.to(device=device, dtype=trunk_dtype)\n        xt_dev = xt_cpu.to(device=device, dtype=trunk_dtype)\n\n        if torch.cuda.is_available() and device.type == "cuda":\n            torch.cuda.synchronize(device)\n        t0 = time.perf_counter()\n\n        if is_x2:\n            out_nchw = head_dev(f_dev, xt_dev, scale=scale)\n        else:\n            out_nchw = head_dev(f_dev, xt_dev)\n\n        if torch.cuda.is_available() and device.type == "cuda":\n            torch.cuda.synchronize(device)\n        dt_ms = (time.perf_counter() - t0) * 1000.0\n\n        rgb = ((out_nchw[:, :3].float() + 1.0) * 0.5).clamp_(0.0, 1.0).movedim(1, -1).contiguous().cpu()\n        pixel_head.to("cpu")\n\n        logger.info(\n            f"[Flux3PixelDecode] Decoded {tuple(rgb.shape[1:3])} (scale={scale}x, last_sigma={sigma:.4f}) in {dt_ms:.2f} ms (Zero VAE)"\n        )\n        return (rgb,)\n\n\nNODE_CLASS_MAPPINGS = {\n    "Flux3PixelHeadLoader": Flux3PixelHeadLoader,\n    "Flux3PixelHeadTap": Flux3PixelHeadTap,\n    "Flux3PixelDecode": Flux3PixelDecode,\n}\n\nNODE_DISPLAY_NAME_MAPPINGS = {\n    "Flux3PixelHeadLoader": "Load FLUX.3 Klein Pixel Head",\n    "Flux3PixelHeadTap": "FLUX.3 Klein Pixel Head Tap",\n    "Flux3PixelDecode": "FLUX.3 Klein Pixel Decode (Zero VAE)",\n}\n', 'README.md': '# ComfyUI-Flux3-PixelHead\n\nZero-VAE Direct RGB Pixel Readout Head (`1x` & `2x`) for **FLUX.3-Image-Klein** (`flux3-image-klein` FlowMap Distilled & `flux3-image-klein-base`) in ComfyUI.\n\nTaps `diffusion_model.final_layer["image"].linear` (`3072`-dim modulated image tokens) + denoised `128`-ch latent `x0_hat_norm` at the final sampling step, decoding directly to RGB in **~4–6 ms** without loading `flux2-vae.safetensors` (`312 MiB`).\n\n## License\nMIT License (100% clean-room implementation).\n', 'tap.py': '"""Non-invasive ComfyUI ModelPatcher tap for FLUX.3-Image-Klein (`comfy.ldm.flux3.model.Flux3`).\n\nLicense: MIT\nCaptures the final denoising step\'s pre-linear hidden states `f` (`[B, H*W, 3072]`) from\n`diffusion_model.final_layer["image"].linear` and the exact CFG-combined denoised latent\n`x0_hat_norm` (`[B, 128, H_lat, W_lat]`), supporting:\n  - Distilled FlowMap (`flux3-image-klein`, 4~16 steps, CFG = 1.0)\n  - Base CFG (`flux3-image-klein-base`, 25~50 steps, CFG > 1.0 via `f_u + cfg * (f_c - f_u)`)\n  - Text-to-Image (T2I), Image-to-Image (I2I), and Multi-Reference Image Editing (MI2I)\n    (`final_layer["image"]` already receives `x[:, :H * W]`, stripping reference & text tokens automatically!)\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Optional\n\nimport comfy.patcher_extension\nimport torch\n\n\nTAP_WRAPPER_KEY = "flux3_pixel_head_tap"\n\n\n@dataclass\nclass StepRecord:\n    sigma: float\n    forwards: list[dict] = field(default_factory=list)\n    cond_scale: Optional[float] = None\n    denoised: Optional[torch.Tensor] = None\n\n\nclass Flux3TapState:\n    """Stores only the latest actual DiT forward step\'s features on CPU."""\n\n    def __init__(self):\n        self.current_step: Optional[StepRecord] = None\n        self.last_valid_step: Optional[StepRecord] = None\n\n    def open_step(self, sigma: float) -> None:\n        self.current_step = StepRecord(sigma=sigma)\n\n    def close_step(self) -> None:\n        if self.current_step is not None and len(self.current_step.forwards) > 0:\n            self.last_valid_step = self.current_step\n        self.current_step = None\n\n    def resolve_final_features(self) -> tuple[torch.Tensor, torch.Tensor, float]:\n        """Return `(f_combined [B, H*W, 3072], x0_hat_norm [B, 128, H_lat, W_lat], sigma)`."""\n        step = self.last_valid_step\n        if step is None or not step.forwards:\n            raise RuntimeError(\n                "Flux3PixelDecode: 未捕获到任何前向特征！请确认已将 `FLUX.3 Pixel Head Tap` "\n                "串联在进入 KSampler 的 MODEL 连线上，且本次运行实际执行了采样步骤。"\n            )\n\n        by_tag: dict[int, list[tuple[torch.Tensor, torch.Tensor]]] = {0: [], 1: []}\n        for rec in step.forwards:\n            cou = rec["cond_or_uncond"]\n            n_chunks = len(cou)\n            f_chunks = rec["f"].chunk(n_chunks, dim=0)\n            d_chunks = rec["denoised"].chunk(n_chunks, dim=0)\n            for tag, fc, dc in zip(cou, f_chunks, d_chunks):\n                by_tag.setdefault(int(tag), []).append((fc, dc))\n\n        if not by_tag.get(0):\n            raise RuntimeError("Flux3PixelDecode: 最后一步未找到正向条件 (cond=0) 特征记录。")\n\n        f_cond = torch.cat([p[0] for p in by_tag[0]], dim=0).float()\n        d_cond = torch.cat([p[1] for p in by_tag[0]], dim=0).float()\n\n        if by_tag.get(1) and step.cond_scale is not None and abs(step.cond_scale - 1.0) > 1e-5:\n            f_uncond = torch.cat([p[0] for p in by_tag[1]], dim=0).float()\n            d_uncond = torch.cat([p[1] for p in by_tag[1]], dim=0).float()\n            scale = float(step.cond_scale)\n            f_final = f_uncond + scale * (f_cond - f_uncond)\n            x0_hat_fallback = d_uncond + scale * (d_cond - d_uncond)\n        else:\n            f_final = f_cond\n            x0_hat_fallback = d_cond\n\n        x0_hat_norm = step.denoised.float() if step.denoised is not None else x0_hat_fallback\n        return f_final, x0_hat_norm, float(step.sigma)\n\n\ndef attach_flux3_tap(model_patcher) -> tuple[object, Flux3TapState]:\n    """Clone `model_patcher` and install lightweight step + `final_layer[\'image\'].linear` pre-hooks."""\n    cloned = model_patcher.clone()\n    state = Flux3TapState()\n\n    cloned.remove_wrappers_with_key(comfy.patcher_extension.WrappersMP.CALC_COND_BATCH, TAP_WRAPPER_KEY)\n    cloned.remove_wrappers_with_key(comfy.patcher_extension.WrappersMP.APPLY_MODEL, TAP_WRAPPER_KEY)\n\n    def calc_cond_batch_wrapper(executor, model, conds, x_in, timestep, model_options):\n        sig = float(timestep.flatten()[0].detach().cpu().item())\n        state.open_step(sig)\n        try:\n            return executor(model, conds, x_in, timestep, model_options)\n        finally:\n            state.close_step()\n\n    def apply_model_wrapper(executor, x, t, c_concat=None, c_crossattn=None, control=None, transformer_options={}, **kwargs):\n        step = state.current_step\n        cou = (transformer_options or {}).get("cond_or_uncond")\n        if step is None or not cou:\n            return executor(x, t, c_concat, c_crossattn, control, transformer_options, **kwargs)\n\n        base_model = executor.class_obj\n        diff_model = getattr(base_model, "diffusion_model", None)\n        final_layer = getattr(diff_model, "final_layer", None)\n        img_head = final_layer["image"] if isinstance(final_layer, (dict, torch.nn.ModuleDict)) and "image" in final_layer else None\n        linear_mod = getattr(img_head, "linear", None)\n        if linear_mod is None:\n            raise RuntimeError(\n                f"Flux3PixelHeadTap: 当前模型 ({type(diff_model).__name__}) 不包含 `final_layer[\'image\'].linear`，请确认底模为 FLUX.3-Image-Klein。"\n            )\n\n        rec = {\n            "denoised": None,\n            "f": None,\n            "cond_or_uncond": list(cou),\n        }\n\n        def _pre_linear_hook(_mod, inputs):\n            if rec["f"] is None and len(inputs) > 0:\n                rec["f"] = inputs[0].detach().to(device="cpu", dtype=torch.bfloat16)\n\n        handle = linear_mod.register_forward_pre_hook(_pre_linear_hook)\n        try:\n            out = executor(x, t, c_concat, c_crossattn, control, transformer_options, **kwargs)\n        finally:\n            handle.remove()\n\n        if rec["f"] is not None:\n            rec["denoised"] = out.detach().to(device="cpu", dtype=torch.float32)\n            step.forwards.append(rec)\n        return out\n\n    def post_cfg_hook(args):\n        target_step = state.current_step if state.current_step is not None else state.last_valid_step\n        if target_step is not None:\n            if "cond_scale" in args:\n                target_step.cond_scale = float(args["cond_scale"])\n            if "denoised" in args and args["denoised"] is not None:\n                target_step.denoised = args["denoised"].detach().to(device="cpu", dtype=torch.float32)\n        return args["denoised"]\n\n    cloned.add_wrapper_with_key(\n        comfy.patcher_extension.WrappersMP.CALC_COND_BATCH,\n        TAP_WRAPPER_KEY,\n        calc_cond_batch_wrapper,\n    )\n    cloned.add_wrapper_with_key(\n        comfy.patcher_extension.WrappersMP.APPLY_MODEL,\n        TAP_WRAPPER_KEY,\n        apply_model_wrapper,\n    )\n    cloned.set_model_sampler_post_cfg_function(post_cfg_hook)\n    cloned.model_options[TAP_WRAPPER_KEY] = state\n    return cloned, state\n'}
for fname, content in FILES.items():
    (LOCAL_SRC / fname).write_text(content, encoding="utf-8")
    (CUSTOM_NODE_DRIVE / fname).write_text(content, encoding="utf-8")

(WF_DRIVE_DIR / "04_Flux3Klein_PixelHead_免VAE极速直出与对比.json").write_text('{\n  "last_node_id": 14,\n  "last_link_id": 14,\n  "nodes": [\n    {\n      "id": 1,\n      "type": "MarkdownNote",\n      "pos": [\n        40,\n        -220\n      ],\n      "size": [\n        1420,\n        180\n      ],\n      "flags": {},\n      "order": 0,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [],\n      "title": "⚡ FLUX.3-Image-Klein PixelHead：免 VAE 毫秒级直出与官方 VAE 同屏对比",\n      "widgets_values": [\n        "### FLUX.3-Image-Klein Direct Pixel Head（Colab L4 24GB 专用）\\n- **默认免 VAE 直出**：采样结束后由 `Flux3PixelDecode` 在 ~5ms 内直接从 3072 维末步特征输出高清 RGB！\\n- **同屏对比**：下方 `VAELoader (flux2-vae)` 与 `VAEDecode` 默认处于 Bypass 状态，选中按 `Ctrl + B` 即可同种子 1:1 对比！"\n      ]\n    },\n    {\n      "id": 2,\n      "type": "UNETLoader",\n      "pos": [\n        40,\n        20\n      ],\n      "size": [\n        430,\n        88\n      ],\n      "order": 1,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "MODEL",\n          "type": "MODEL",\n          "links": [\n            1\n          ]\n        }\n      ],\n      "widgets_values": [\n        "flux3-image-klein-int8-convrot.safetensors",\n        "default"\n      ]\n    },\n    {\n      "id": 3,\n      "type": "CLIPLoader",\n      "pos": [\n        40,\n        145\n      ],\n      "size": [\n        430,\n        98\n      ],\n      "order": 2,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "CLIP",\n          "type": "CLIP",\n          "links": [\n            3,\n            4\n          ]\n        }\n      ],\n      "widgets_values": [\n        "qwen3vl_4b_fp8_scaled.safetensors",\n        "flux3",\n        "default"\n      ]\n    },\n    {\n      "id": 4,\n      "type": "ModelSamplingFlux",\n      "pos": [\n        40,\n        280\n      ],\n      "size": [\n        430,\n        120\n      ],\n      "order": 3,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 1\n        }\n      ],\n      "outputs": [\n        {\n          "name": "MODEL",\n          "type": "MODEL",\n          "links": [\n            2\n          ]\n        }\n      ],\n      "widgets_values": [\n        2.77259,\n        2.77259,\n        1024,\n        1024\n      ]\n    },\n    {\n      "id": 5,\n      "type": "Flux3PixelHeadLoader",\n      "pos": [\n        40,\n        440\n      ],\n      "size": [\n        430,\n        88\n      ],\n      "order": 4,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "pixel_head",\n          "type": "FLUX3_PIXEL_HEAD",\n          "links": [\n            9\n          ]\n        }\n      ],\n      "widgets_values": [\n        "flux3_klein_pixel_head.safetensors",\n        "bf16_trunk_fp32_out"\n      ]\n    },\n    {\n      "id": 6,\n      "type": "Flux3PixelHeadTap",\n      "pos": [\n        40,\n        565\n      ],\n      "size": [\n        430,\n        68\n      ],\n      "order": 5,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 2\n        }\n      ],\n      "outputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "links": [\n            5\n          ]\n        },\n        {\n          "name": "pixel_tap",\n          "type": "FLUX3_TAP",\n          "links": [\n            10\n          ]\n        }\n      ]\n    },\n    {\n      "id": 7,\n      "type": "CLIPTextEncode",\n      "pos": [\n        510,\n        20\n      ],\n      "size": [\n        430,\n        180\n      ],\n      "order": 6,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "clip",\n          "type": "CLIP",\n          "link": 3\n        }\n      ],\n      "outputs": [\n        {\n          "name": "CONDITIONING",\n          "type": "CONDITIONING",\n          "links": [\n            6\n          ]\n        }\n      ],\n      "widgets_values": [\n        "Crisp 2D anime illustration of a silver-haired swordswoman standing on a rain-slicked neon rooftop at dusk, sharp clean black linework, vibrant reflections, fine fabric folds, detailed eyes."\n      ]\n    },\n    {\n      "id": 8,\n      "type": "CLIPTextEncode",\n      "pos": [\n        510,\n        235\n      ],\n      "size": [\n        430,\n        90\n      ],\n      "order": 7,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "clip",\n          "type": "CLIP",\n          "link": 4\n        }\n      ],\n      "outputs": [\n        {\n          "name": "CONDITIONING",\n          "type": "CONDITIONING",\n          "links": [\n            7\n          ]\n        }\n      ],\n      "widgets_values": [\n        ""\n      ]\n    },\n    {\n      "id": 9,\n      "type": "EmptyFlux2LatentImage",\n      "pos": [\n        510,\n        360\n      ],\n      "size": [\n        430,\n        108\n      ],\n      "order": 8,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "LATENT",\n          "type": "LATENT",\n          "links": [\n            8\n          ]\n        }\n      ],\n      "widgets_values": [\n        1024,\n        1024,\n        1\n      ]\n    },\n    {\n      "id": 10,\n      "type": "KSampler",\n      "pos": [\n        980,\n        20\n      ],\n      "size": [\n        340,\n        265\n      ],\n      "order": 9,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 5\n        },\n        {\n          "name": "positive",\n          "type": "CONDITIONING",\n          "link": 6\n        },\n        {\n          "name": "negative",\n          "type": "CONDITIONING",\n          "link": 7\n        },\n        {\n          "name": "latent_image",\n          "type": "LATENT",\n          "link": 8\n        }\n      ],\n      "outputs": [\n        {\n          "name": "LATENT",\n          "type": "LATENT",\n          "links": [\n            11,\n            12\n          ]\n        }\n      ],\n      "widgets_values": [\n        20261122,\n        "fixed",\n        16,\n        1.0,\n        "euler",\n        "simple",\n        1.0\n      ]\n    },\n    {\n      "id": 11,\n      "type": "Flux3PixelDecode",\n      "pos": [\n        980,\n        325\n      ],\n      "size": [\n        340,\n        110\n      ],\n      "order": 10,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "samples",\n          "type": "LATENT",\n          "link": 11\n        },\n        {\n          "name": "pixel_head",\n          "type": "FLUX3_PIXEL_HEAD",\n          "link": 9\n        },\n        {\n          "name": "pixel_tap",\n          "type": "FLUX3_TAP",\n          "link": 10\n        }\n      ],\n      "outputs": [\n        {\n          "name": "image",\n          "type": "IMAGE",\n          "links": [\n            13\n          ]\n        }\n      ],\n      "widgets_values": [\n        "auto"\n      ]\n    },\n    {\n      "id": 12,\n      "type": "SaveImage",\n      "pos": [\n        1360,\n        20\n      ],\n      "size": [\n        420,\n        415\n      ],\n      "order": 11,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "images",\n          "type": "IMAGE",\n          "link": 13\n        }\n      ],\n      "outputs": [],\n      "widgets_values": [\n        "Flux3Klein_PixelHead_ZeroVAE"\n      ]\n    },\n    {\n      "id": 13,\n      "type": "VAELoader",\n      "pos": [\n        510,\n        515\n      ],\n      "size": [\n        430,\n        68\n      ],\n      "order": 12,\n      "mode": 4,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "VAE",\n          "type": "VAE",\n          "links": [\n            14\n          ]\n        }\n      ],\n      "widgets_values": [\n        "flux2-vae.safetensors"\n      ]\n    },\n    {\n      "id": 14,\n      "type": "VAEDecode",\n      "pos": [\n        980,\n        515\n      ],\n      "size": [\n        340,\n        68\n      ],\n      "order": 13,\n      "mode": 4,\n      "inputs": [\n        {\n          "name": "samples",\n          "type": "LATENT",\n          "link": 12\n        },\n        {\n          "name": "vae",\n          "type": "VAE",\n          "link": 14\n        }\n      ],\n      "outputs": [\n        {\n          "name": "IMAGE",\n          "type": "IMAGE",\n          "links": []\n        }\n      ]\n    }\n  ],\n  "links": [\n    [\n      1,\n      2,\n      0,\n      4,\n      0,\n      "MODEL"\n    ],\n    [\n      2,\n      4,\n      0,\n      6,\n      0,\n      "MODEL"\n    ],\n    [\n      3,\n      3,\n      0,\n      7,\n      0,\n      "CLIP"\n    ],\n    [\n      4,\n      3,\n      0,\n      8,\n      0,\n      "CLIP"\n    ],\n    [\n      5,\n      6,\n      0,\n      10,\n      0,\n      "MODEL"\n    ],\n    [\n      6,\n      7,\n      0,\n      10,\n      1,\n      "CONDITIONING"\n    ],\n    [\n      7,\n      8,\n      0,\n      10,\n      2,\n      "CONDITIONING"\n    ],\n    [\n      8,\n      9,\n      0,\n      10,\n      3,\n      "LATENT"\n    ],\n    [\n      9,\n      5,\n      0,\n      11,\n      1,\n      "FLUX3_PIXEL_HEAD"\n    ],\n    [\n      10,\n      6,\n      1,\n      11,\n      2,\n      "FLUX3_TAP"\n    ],\n    [\n      11,\n      10,\n      0,\n      11,\n      0,\n      "LATENT"\n    ],\n    [\n      12,\n      10,\n      0,\n      14,\n      0,\n      "LATENT"\n    ],\n    [\n      13,\n      11,\n      0,\n      12,\n      0,\n      "IMAGE"\n    ],\n    [\n      14,\n      13,\n      0,\n      14,\n      1,\n      "VAE"\n    ]\n  ],\n  "version": 0.4\n}', encoding="utf-8")
print("Google Drive 挂载成功，ComfyUI-Flux3-PixelHead 插件与对比工作流已同步至:", CUSTOM_NODE_DRIVE)

CHECK_ITEMS = [
    ("models/diffusion_models", "flux3-image-klein-int8-convrot.safetensors"),
    ("models/text_encoders", "qwen3vl_4b_fp8_scaled.safetensors"),
    ("models/vae", "flux2-vae.safetensors"),
    ("models/vae", "full_encoder_small_decoder.safetensors"),
]
for sub, fn in CHECK_ITEMS:
    p = DRIVE_HUB / sub / fn
    st = f"✅ 网盘已有 ({p.stat().st_size/1e9:.2f} GB)" if p.exists() else "⬇️ 将在单元格 2 自动从无门控高速镜像拉取"
    print(f"   - {sub}/{fn}: {st}")

In [ ]:
# @title 2. 升级最新 ComfyUI 引擎并将 FLUX.3 Klein 与官方 Teacher VAE 预热至本地 NVMe SSD
import json, os, shutil, struct, subprocess, sys, time
from pathlib import Path

t0 = time.time()
DRIVE_HUB = Path("/content/drive/MyDrive/ComfyUI_Hub")
COMFY_DIR = Path("/content/ComfyUI")
LOCAL_MODELS = Path("/content/local_models")
LOCAL_MODELS.mkdir(parents=True, exist_ok=True)

# @markdown ### 训练覆盖模式（Teacher VAE 严格固定为官方原版 `flux2-vae.safetensors`）：
# @markdown - `"both_alternating"`：同时加载 Distilled (FlowMap 4~16步) 与 Base (25~50步 CFG)，单头全兼容（推荐，共 ~10.8 GB）
# @markdown - `"distilled_only"`：仅加载 `flux3-image-klein-int8-convrot`（最快）
DIT_TRAIN_MODE = "both_alternating"  # @param ["both_alternating", "distilled_only"]

def get_hf_headers() -> list[str]:
    tok = os.environ.get("HF_TOKEN", "").strip()
    if not tok:
        try:
            from google.colab import userdata
            tok = (userdata.get("HF_TOKEN") or "").strip()
        except Exception:
            tok = ""
    return [f"--header=Authorization: Bearer {tok}"] if tok else []


def ensure_fresh_comfyui(required_rel_file: str) -> None:
    req_path = COMFY_DIR / required_rel_file
    if not COMFY_DIR.exists():
        # Prefer extracting an existing Hub tarball if available, then fast-forward
        candidates = sorted(DRIVE_HUB.glob("ComfyUI_*_bundle.tar.gz"))
        if candidates:
            print(f"从 Drive 极速解压 ComfyUI 基础包: {candidates[0].name} ...")
            subprocess.run(["tar", "-xzf", str(candidates[0]), "-C", "/content/"], check=True)
        else:
            print("📥 克隆官方最新 ComfyUI...")
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(COMFY_DIR)], check=True)
    if not req_path.exists():
        print(f"升级 ComfyUI 至最新主干以启用 `{required_rel_file}`...")
        subprocess.run(["git", "-C", str(COMFY_DIR), "fetch", "--depth", "1", "origin", "master"], check=True)
        subprocess.run(["git", "-C", str(COMFY_DIR), "reset", "--hard", "FETCH_HEAD"], check=True)

    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    print("安装/校验 ComfyUI 与评测依赖 (lpips, pyarrow, safetensors)...")
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", str(COMFY_DIR / "requirements.txt")], check=False)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "lpips", "pyarrow", "safetensors", "huggingface_hub", "einops"], check=True)
    if shutil.which("aria2c") is None:
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], check=False)


def is_valid_safetensors(p: Path, min_mb: float = 5.0) -> bool:
    if not p.exists() or p.stat().st_size < min_mb * 1024 * 1024:
        return False
    try:
        with open(p, "rb") as f:
            n = struct.unpack("<Q", f.read(8))[0]
            if n <= 0 or n > 100 * 1024 * 1024:
                return False
            hdr = json.loads(f.read(n))
        max_end = max((v["data_offsets"][1] for k, v in hdr.items() if k != "__metadata__"), default=0)
        return (8 + n + max_end) == p.stat().st_size
    except Exception:
        return False


def stage_model(rel_sub: str, fname: str, hf_urls: list[str] | str, min_mb: float, save_to_drive: bool = True, alt_drive_names: list[str] = []) -> Path:
    if isinstance(hf_urls, str):
        hf_urls = [hf_urls]
    local_p = LOCAL_MODELS / rel_sub / fname
    local_p.parent.mkdir(parents=True, exist_ok=True)
    if is_valid_safetensors(local_p, min_mb):
        print(f"  NVMe 已就绪: {fname} ({local_p.stat().st_size/1e9:.2f} GB)")
        return local_p

    drive_dir = DRIVE_HUB / "models" / rel_sub
    drive_dir.mkdir(parents=True, exist_ok=True)
    for cand_name in [fname] + list(alt_drive_names):
        cand_p = drive_dir / cand_name
        if is_valid_safetensors(cand_p, min_mb):
            dst_p = LOCAL_MODELS / rel_sub / cand_name
            if not is_valid_safetensors(dst_p, min_mb):
                print(f"  从 Drive 顺序预热至本地 NVMe: {cand_name} ({cand_p.stat().st_size/1e9:.2f} GB) ...")
                shutil.copyfile(cand_p, dst_p)
            return dst_p

    hdr_args = get_hf_headers()
    last_err = None
    for url in hf_urls:
        try:
            print(f"  本地 NVMe 高速直拉: {fname} <- {url.split('/resolve/')[0]} ...")
            tmp_dl = local_p.with_suffix(".dl")
            cmd = [
                "aria2c", "--console-log-level=warn", "--file-allocation=none",
                "-c", "-x", "16", "-s", "16", "-k", "1M",
                *hdr_args,
                url, "-d", str(local_p.parent), "-o", tmp_dl.name,
            ]
            subprocess.run(cmd, check=True)
            os.replace(tmp_dl, local_p)
            assert is_valid_safetensors(local_p, min_mb), f"下载文件头校验失败: {local_p}"
            if save_to_drive:
                shutil.copyfile(local_p, drive_dir / fname)
            return local_p
        except Exception as e:
            last_err = e
            print(f"  ⚠源 {url} 拉取失败，尝试备用源: {e}")
    raise RuntimeError(f"无法获取模型 {fname}: {last_err}")

ensure_fresh_comfyui("comfy/ldm/flux3/model.py")

DIT_DISTILLED_PATH = stage_model(
    "diffusion_models",
    "flux3-image-klein-int8-convrot.safetensors",
    [
        "https://huggingface.co/ml-resources/flux-3-image-klein-comfy/resolve/main/diffusion_models/flux3-image-klein-int8-convrot.safetensors",
        "https://huggingface.co/black-forest-labs/flux-3-image-klein-comfy/resolve/main/diffusion_models/flux3-image-klein-int8-convrot.safetensors",
    ],
    min_mb=4800,
    save_to_drive=True,
)

DIT_BASE_PATH = None
if DIT_TRAIN_MODE == "both_alternating":
    DIT_BASE_PATH = stage_model(
        "diffusion_models",
        "flux3-image-klein-base-int8-convrot.safetensors",
        [
            "https://huggingface.co/ml-resources/flux-3-image-klein-comfy/resolve/main/diffusion_models/flux3-image-klein-base-int8-convrot.safetensors",
            "https://huggingface.co/black-forest-labs/flux-3-image-klein-comfy/resolve/main/diffusion_models/flux3-image-klein-base-int8-convrot.safetensors",
        ],
        min_mb=4800,
        save_to_drive=False,
    )

# 直接复用你网盘已有的 Qwen3-VL-4B 权重（0 下载！）
TE_PATH = stage_model(
    "text_encoders",
    "qwen3vl_4b_fp8_scaled.safetensors",
    [
        "https://huggingface.co/Comfy-Org/Krea-2/resolve/main/text_encoders/qwen3vl_4b_fp8_scaled.safetensors",
        "https://huggingface.co/ml-resources/flux-3-image-klein-comfy/resolve/main/text_encoders/qwen3vl_4b_int8_convrot.safetensors",
    ],
    min_mb=4000,
    alt_drive_names=["qwen3vl_4b_fp8_scaled.safetensors", "qwen3vl_4b_int8_convrot.safetensors"],
)

# 官方原版 Teacher VAE: flux2-vae.safetensors（网盘已就绪！）
VAE_PATH = stage_model(
    "vae",
    "flux2-vae.safetensors",
    [
        "https://huggingface.co/ml-resources/flux-3-image-klein-comfy/resolve/main/vae/flux2-vae.safetensors",
        "https://huggingface.co/Comfy-Org/flux2-dev/resolve/main/split_files/vae/flux2-vae.safetensors",
    ],
    min_mb=250,
)

# 若网盘已有 BFL 官方轻量解码器，同步预热供单元格 5 同台跑分打擂
SMALL_DEC_DRIVE = DRIVE_HUB / "models/vae/full_encoder_small_decoder.safetensors"
SMALL_DEC_PATH = LOCAL_MODELS / "vae/full_encoder_small_decoder.safetensors"
if SMALL_DEC_DRIVE.exists() and not SMALL_DEC_PATH.exists():
    shutil.copyfile(SMALL_DEC_DRIVE, SMALL_DEC_PATH)

for p in (str(COMFY_DIR), "/content"):
    if p not in sys.path:
        sys.path.insert(0, p)

import folder_paths
for k in ("diffusion_models", "text_encoders", "vae"):
    folder_paths.add_model_folder_path(k, str(LOCAL_MODELS / k))

print(f"\nFLUX.3-Image-Klein 环境与 NVMe 预热完成。耗时: {time.time() - t0:.1f} 秒")

In [ ]:
# @title 3. 构建多领域高锐度校准图集并离线预编码 128ch 潜变量与文本嵌入（随后卸载文本塔）
import gc, io, json, random, time, urllib.parse, urllib.request
from pathlib import Path
import numpy as np
import pyarrow.parquet as pq
import torch
from PIL import Image, ImageDraw, ImageFont
import comfy.model_management, comfy.sd, comfy.utils

CACHE_ROOT = Path("/content/flux3_ph_cache")
IMG_DIR, LAT_DIR, COND_DIR = CACHE_ROOT / "images", CACHE_ROOT / "latents", CACHE_ROOT / "conds"
MANIFEST_PATH = CACHE_ROOT / "manifest.json"
for d in (IMG_DIR, LAT_DIR, COND_DIR):
    d.mkdir(parents=True, exist_ok=True)

TOTAL_SAMPLES = 1200  # @param {type:"integer"}
CALIB_RES = 768       # @param [768, 1024] {type:"raw"}

def center_crop_resize(im: Image.Image, size: int) -> Image.Image:
    im = im.convert("RGB")
    w, h = im.size
    side = min(w, h)
    left = (w - side) // 2
    top = (h - side) // 2
    return im.crop((left, top, left + side, top + side)).resize((size, size), Image.Resampling.LANCZOS)


def make_synthetic_cel_manga_typo(idx: int, size: int, title_banner: str) -> tuple[Image.Image, str]:
    rng = random.Random(20261122 + idx)
    sub_kind = idx % 3
    if sub_kind == 0:
        bg_c = (rng.randint(215, 250), rng.randint(220, 250), rng.randint(230, 255))
        im = Image.new("RGB", (size, size), bg_c)
        dr = ImageDraw.Draw(im)
        for _ in range(14):
            x0, y0 = rng.randint(0, size - 120), rng.randint(0, size - 120)
            x1, y1 = x0 + rng.randint(80, 360), y0 + rng.randint(80, 360)
            fill = (rng.randint(30, 245), rng.randint(30, 245), rng.randint(30, 245))
            outline = (18, 18, 24)
            w_line = rng.choice([2, 3, 4])
            if rng.random() < 0.5:
                dr.ellipse([x0, y0, x1, y1], fill=fill, outline=outline, width=w_line)
            else:
                dr.rounded_rectangle([x0, y0, x1, y1], radius=18, fill=fill, outline=outline, width=w_line)
        return im, "anime_cel"
    elif sub_kind == 1:
        arr = np.full((size, size, 3), 248, dtype=np.uint8)
        yy, xx = np.mgrid[0:size, 0:size]
        period = rng.choice([4, 6, 8])
        dots = (((xx % period == 0) & (yy % period == 0))).astype(np.uint8)
        arr[size // 6 : size // 2, size // 6 : size * 5 // 6] -= dots[size // 6 : size // 2, size // 6 : size * 5 // 6, None] * 170
        im = Image.fromarray(arr, mode="RGB")
        dr = ImageDraw.Draw(im)
        for _ in range(35):
            p0 = (rng.randint(0, size), rng.randint(0, size))
            p1 = (rng.randint(0, size), rng.randint(0, size))
            dr.line([p0, p1], fill=(12, 12, 12), width=rng.choice([1, 2, 3]))
        return im, "manga_line"
    else:
        dark = rng.random() < 0.5
        bg = (18, 20, 28) if dark else (246, 244, 238)
        fg = (242, 245, 250) if dark else (20, 22, 28)
        accent = (rng.randint(180, 255), rng.randint(60, 190), rng.randint(50, 220))
        im = Image.new("RGB", (size, size), bg)
        dr = ImageDraw.Draw(im)
        dr.rectangle([36, 36, size - 36, size - 36], outline=accent, width=4)
        sample_lines = [
            title_banner,
            "FastFrames Zero-VAE Readout 2026",
            "0123456789  PSNR 35.8 dB  5.0 ms",
            "Sharp Edges · Clean Cel · Fine Detail",
            "abcdefghijklmnopqrstuvwxyz ABCDEFG",
        ]
        y_cur = 75
        for li, text in enumerate(sample_lines):
            fsize = [40, 28, 24, 20, 16][li % 5]
            try:
                font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", fsize)
            except Exception:
                font = ImageFont.load_default()
            dr.text((60, y_cur), text, fill=fg if li % 2 == 0 else accent, font=font)
            y_cur += fsize + 36
        return im, "typography"


def fetch_hf_parquet_pil(dataset_id: str, max_count: int, min_side: int = 384) -> list[Image.Image]:
    pil_list = []
    shard_urls = []
    try:
        pq_api = f"https://huggingface.co/api/datasets/{dataset_id}/parquet"
        req = urllib.request.Request(pq_api, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=25) as r:
            pq_tree = json.loads(r.read().decode("utf-8"))
        for cfg_map in pq_tree.values():
            if isinstance(cfg_map, dict):
                for split_list in cfg_map.values():
                    if isinstance(split_list, list):
                        shard_urls.extend(split_list)
    except Exception:
        pass
    if not shard_urls:
        api_url = f"https://huggingface.co/api/datasets/{dataset_id}"
        req = urllib.request.Request(api_url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=25) as r:
            info = json.loads(r.read().decode("utf-8"))
        for s in info.get("siblings", []):
            rf = s.get("rfilename", "")
            if rf.endswith(".parquet"):
                shard_urls.append(f"https://huggingface.co/datasets/{dataset_id}/resolve/main/{urllib.parse.quote(rf)}")

    for idx_sh, sh_url in enumerate(shard_urls):
        if len(pil_list) >= max_count:
            break
        local_pq = CACHE_ROOT / f"_tmp_{dataset_id.replace('/', '_')}_{idx_sh:03d}.parquet"
        if not local_pq.exists():
            print(f"  拉取公开无门控校准分片 [{idx_sh}]: {dataset_id} ...")
            urllib.request.urlretrieve(sh_url, str(local_pq))
        table = pq.read_table(local_pq)
        img_col = next((c for c in table.column_names if "image" in c.lower()), None)
        if img_col is None:
            local_pq.unlink(missing_ok=True)
            continue
        for row_val in table[img_col].to_pylist():
            if len(pil_list) >= max_count:
                break
            raw_bytes = row_val.get("bytes") if isinstance(row_val, dict) else None
            if not raw_bytes:
                continue
            try:
                im = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
                if min(im.size) < min_side:
                    continue
                pil_list.append(im)
            except Exception:
                continue
        local_pq.unlink(missing_ok=True)
    return pil_list

if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    print(f"检测到已完成预编码的本地缓存 ({len(manifest)} 条样本)，直接复用！")
else:
    t_c = time.time()
    n_art = int(TOTAL_SAMPLES * 0.42)
    n_photo = int(TOTAL_SAMPLES * 0.38)
    n_syn = TOTAL_SAMPLES - n_art - n_photo

    rows = []
    print(f"[1/3] 构建多领域校准图像 ({TOTAL_SAMPLES} 张 @ {CALIB_RES}x{CALIB_RES})...")
    for im in fetch_hf_parquet_pil("huggan/wikiart", n_art):
        cid = len(rows)
        fn = f"{cid:05d}_art_illust.png"
        center_crop_resize(im, CALIB_RES).save(IMG_DIR / fn)
        rows.append({"id": cid, "image": fn, "domain": "art_illust"})
    for im in fetch_hf_parquet_pil("detection-datasets/coco", n_photo):
        cid = len(rows)
        fn = f"{cid:05d}_real_photo.png"
        center_crop_resize(im, CALIB_RES).save(IMG_DIR / fn)
        rows.append({"id": cid, "image": fn, "domain": "real_photo"})
    for k in range(n_syn):
        cid = len(rows)
        im_s, sub_tag = make_synthetic_cel_manga_typo(k, CALIB_RES, "FLUX.3 KLEIN PIXEL HEAD")
        fn = f"{cid:05d}_{sub_tag}.png"
        im_s.save(IMG_DIR / fn)
        rows.append({"id": cid, "image": fn, "domain": sub_tag})

    rng_split = random.Random(42)
    idxs = list(range(len(rows)))
    rng_split.shuffle(idxs)
    test_set = set(idxs[: min(80, max(24, len(rows) // 12))])
    for i, r in enumerate(rows):
        r["split"] = "test" if i in test_set else "train"

    print(f"[2/3] 使用官方 Teacher VAE ({VAE_PATH.name}) 预编码 {len(rows)} 张 128ch 潜变量...")
    vae = comfy.sd.VAE(sd=comfy.utils.load_torch_file(str(VAE_PATH)))
    for i, r in enumerate(rows):
        im = Image.open(IMG_DIR / r["image"]).convert("RGB")
        arr = torch.from_numpy(np.array(im)).float().div_(255.0).unsqueeze(0)
        with torch.no_grad():
            z = vae.encode(arr)  # [1, 128, H/16, W/16]
        lat_fn = f"{r['id']:05d}.pt"
        torch.save(z.to(torch.bfloat16).cpu(), LAT_DIR / lat_fn)
        r["latent"] = lat_fn
        if (i + 1) % 200 == 0 or (i + 1) == len(rows):
            print(f"    已编码 Latent: {i + 1}/{len(rows)}")
    del vae
    comfy.model_management.cleanup_models_gc(); gc.collect(); torch.cuda.empty_cache()

    print("[3/3] 使用 CLIPType.FLUX3 预编码条件嵌入并卸载文本编码器...")
    PROMPTS = {
        "art_illust": [
            "High-resolution illustration with expressive brushwork, clean color transitions, rich artistic detail.",
            "2D anime illustration, crisp cel shading, clean linework, balanced color palette.",
        ],
        "real_photo": [
            "Sharp high-resolution photograph, natural lighting, realistic surface textures and depth of field.",
            "Candid outdoor and indoor photography, accurate colors, crisp edges and fine details.",
        ],
        "anime_cel": ["Clean 2D cel-shaded anime art with bold black ink outlines and smooth flat color blocks."],
        "manga_line": ["Monochrome manga page with crisp black ink lineart and fine halftone screentone patterns."],
        "typography": ["Graphic poster design featuring razor-sharp typography, crisp lettering, and geometric borders."],
    }
    clip = comfy.sd.load_clip(ckpt_paths=[str(TE_PATH)], embedding_directory=None, clip_type=comfy.sd.CLIPType.FLUX3)
    cond_map, cid_counter = {}, 0
    for dom, plist in PROMPTS.items():
        cond_map[dom] = []
        for ptxt in plist:
            cond = clip.encode_from_tokens_scheduled(clip.tokenize(ptxt))
            cfn = f"cond_{cid_counter:03d}.pt"
            torch.save(cond[0][0].detach().to(torch.bfloat16).cpu(), COND_DIR / cfn)
            cond_map[dom].append(cfn)
            cid_counter += 1
    for r in rows:
        opts = cond_map.get(r["domain"], cond_map["real_photo"])
        r["cond"] = opts[r["id"] % len(opts)]

    del clip
    comfy.model_management.unload_all_models(); comfy.model_management.soft_empty_cache(); gc.collect(); torch.cuda.empty_cache()
    MANIFEST_PATH.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    manifest = rows
    print(f"预编码完成。耗时 {time.time() - t_c:.1f}s | 空闲显存: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

In [ ]:
# @title 4. P0 冒烟自检 + 3000 Rollouts 双模式交替正式训练（自动同步回 Google Drive）
from collections import deque
import gc, importlib.util, json, math, random, shutil, time
from pathlib import Path
import torch
import comfy.model_management, comfy.sd, comfy.utils

LOCAL_MODELS = Path("/content/local_models")
if "DIT_DISTILLED_PATH" not in globals():
    DIT_DISTILLED_PATH = LOCAL_MODELS / "diffusion_models/flux3-image-klein-int8-convrot.safetensors"
if "DIT_BASE_PATH" not in globals():
    cand_b = LOCAL_MODELS / "diffusion_models/flux3-image-klein-base-int8-convrot.safetensors"
    DIT_BASE_PATH = cand_b if cand_b.exists() else None
if "VAE_PATH" not in globals():
    VAE_PATH = LOCAL_MODELS / "vae/flux2-vae.safetensors"

spec = importlib.util.spec_from_file_location("flux3_pixel_head", "/content/ComfyUI-Flux3-PixelHead/flux3_pixel_head.py")
ph_mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(ph_mod)
Flux3PixelHead = ph_mod.Flux3PixelHead
save_flux3_pixel_head = ph_mod.save_flux3_pixel_head
load_flux3_pixel_head = ph_mod.load_flux3_pixel_head

TOTAL_ROLLOUTS = 3000        # @param {type:"integer"}
REPLAY_BUFFER_SIZE = 64      # @param {type:"integer"}
UPDATES_PER_ROLLOUT = 16     # @param {type:"integer"}
WINDOW_TOKENS = 32           # @param {type:"integer"}
MARGIN_TOKENS = 2            # @param {type:"integer"}
HEAD_WIDTH = 1024            # @param [768, 1024] {type:"raw"}
LEARNING_RATE = 3e-4         # @param {type:"number"}
WARMUP_STEPS = 200           # @param {type:"integer"}
SAVE_EVERY_ROLLOUTS = 500    # @param {type:"integer"}
RESUME_IF_EXISTS = True      # @param {type:"boolean"}

DEVICE = torch.device("cuda")
CKPT_DRIVE = Path("/content/drive/MyDrive/FastFrames_Flux3Klein/checkpoints")
PIXEL_HEADS_DRIVE = Path("/content/drive/MyDrive/ComfyUI_Hub/models/pixel_heads")
VAE_DRIVE = Path("/content/drive/MyDrive/ComfyUI_Hub/models/vae")

manifest = json.loads(Path("/content/flux3_ph_cache/manifest.json").read_text(encoding="utf-8"))
train_rows = [r for r in manifest if r["split"] == "train"]
test_rows = [r for r in manifest if r["split"] == "test"]
random.Random(20261122).shuffle(train_rows)

comfy.model_management.unload_all_models(); gc.collect(); torch.cuda.empty_cache()
print("加载冻结的 FLUX.3-Image-Klein DiT 与官方 Teacher VAE...")
patcher_distilled = comfy.sd.load_diffusion_model(str(DIT_DISTILLED_PATH))
patcher_base = comfy.sd.load_diffusion_model(str(DIT_BASE_PATH)) if DIT_BASE_PATH is not None else patcher_distilled
teacher_vae = comfy.sd.VAE(sd=comfy.utils.load_torch_file(str(VAE_PATH)))

# 精确对齐 FLUX.3 两种 shift (alpha=16 -> shift=2.77259, 默认 shift=1.93586) 下的末步 sigma 分布：
# - Distilled FlowMap (4~16 steps): 末步覆盖 [0.28, 0.72]（官方 16步 alpha=16 末步精确为 0.51613，默认 shift 为 0.31601）
# - Base (25~50 steps): 末步覆盖 [0.10, 0.38]（官方 50步 alpha=16 末步为 0.24615）
DISTILLED_SIGMA_RANGE = (0.280, 0.720)
BASE_SIGMA_RANGE = (0.100, 0.380)


class Flux3RolloutEngine:
    def __init__(self, p_dist, p_base, vae, cache_dir: Path):
        self.p_dist = p_dist
        self.p_base = p_base
        self.vae = vae
        self.lat_dir = cache_dir / "latents"
        self.cond_dir = cache_dir / "conds"
        self.cond_mem = {}

    def _get_cond(self, fn: str) -> torch.Tensor:
        if fn not in self.cond_mem:
            self.cond_mem[fn] = torch.load(self.cond_dir / fn, map_location="cpu", weights_only=True)
        return self.cond_mem[fn]

    @torch.no_grad()
    def rollout(self, row: dict, sigma: float, use_distilled: bool, seed: int):
        patcher = self.p_dist if use_distilled else self.p_base
        comfy.model_management.load_models_gpu([patcher])
        bmodel = patcher.model
        lat_fmt = bmodel.latent_format

        x0_raw = torch.load(self.lat_dir / row["latent"], map_location=DEVICE, weights_only=True).float()
        x0_norm = lat_fmt.process_in(x0_raw)
        cond = self._get_cond(row["cond"]).to(device=DEVICE, dtype=bmodel.get_dtype_inference())

        gen = torch.Generator(device=DEVICE).manual_seed(int(seed))
        xt_norm = (1.0 - sigma) * x0_norm + sigma * torch.randn(x0_norm.shape, generator=gen, device=DEVICE)
        sig_t = torch.tensor([sigma], device=DEVICE, dtype=torch.float32)
        # FlowMap 蒸馏模型在末步读取 sample_sigmas[i+1] = 0.0 作为跳步终点
        topts = {"sample_sigmas": torch.tensor([sigma, 0.0], device=DEVICE, dtype=torch.float32)}

        captured = {}
        lin_mod = bmodel.diffusion_model.final_layer["image"].linear
        def _pre_hk(_m, inp):
            if "f" not in captured:
                captured["f"] = inp[0].detach()
        handle = lin_mod.register_forward_pre_hook(_pre_hk)
        try:
            x0_hat_norm = bmodel.apply_model(xt_norm, sig_t, c_crossattn=cond, transformer_options=topts).float()
        finally:
            handle.remove()

        f = captured["f"].float()  # [1, gh*gw, 3072]
        x0_hat_raw = lat_fmt.process_out(x0_hat_norm)
        rgb_01 = self.vae.decode(x0_hat_raw.cpu())[..., :3]
        if rgb_01.ndim == 5:
            rgb_01 = rgb_01[:, 0]
        tgt_nchw = (rgb_01[0].permute(2, 0, 1).float() * 2.0 - 1.0).clamp_(-1.0, 1.0)

        gh, gw = x0_hat_norm.shape[-2], x0_hat_norm.shape[-1]
        return (
            f[0].view(gh, gw, -1).to(torch.bfloat16).cpu(),
            x0_hat_norm[0].to(torch.bfloat16).cpu(),
            tgt_nchw.to(torch.bfloat16).cpu(),
        )


engine = Flux3RolloutEngine(patcher_distilled, patcher_base, teacher_vae, Path("/content/flux3_ph_cache"))


def pixel_head_loss(pred: torch.Tensor, tgt: torch.Tensor) -> tuple[torch.Tensor, float]:
    l1 = (pred - tgt).abs().mean()
    mse = ((pred - tgt) ** 2).mean()
    gh = (pred.diff(dim=-2) - tgt.diff(dim=-2)).abs().mean()
    gw = (pred.diff(dim=-1) - tgt.diff(dim=-1)).abs().mean()
    total = 0.5 * l1 + 5.0 * mse + 0.05 * (gh + gw)
    mse_01 = max(1e-10, float(mse.detach().item()) * 0.25)
    return total, -10.0 * math.log10(mse_01)


def random_token_window(f_grid, xt_c, tgt_c, win_tok: int, mar_tok: int, rng: random.Random):
    gh, gw = f_grid.shape[:2]
    span = min(gh, gw, win_tok + 2 * mar_tok)
    y0, x0 = rng.randint(0, gh - span), rng.randint(0, gw - span)
    mt = 0 if y0 == 0 else mar_tok
    mb = 0 if (y0 + span) == gh else mar_tok
    ml = 0 if x0 == 0 else mar_tok
    mr = 0 if (x0 + span) == gw else mar_tok
    return (
        f_grid[y0 : y0 + span, x0 : x0 + span].unsqueeze(0),
        xt_c[:, y0 : y0 + span, x0 : x0 + span].unsqueeze(0),
        tgt_c[:, y0 * 16 : (y0 + span) * 16, x0 * 16 : (x0 + span) * 16].unsqueeze(0),
        (mt * 16, mb * 16, ml * 16, mr * 16),
    )


def strip_px_margin(img: torch.Tensor, mar_px: tuple[int, int, int, int]) -> torch.Tensor:
    mt, mb, ml, mr = mar_px
    h, w = img.shape[-2], img.shape[-1]
    return img[..., mt : h - mb, ml : w - mr]


latest_ckpt = CKPT_DRIVE / "latest_state.pt"
if not (RESUME_IF_EXISTS and latest_ckpt.exists()):
    print("\n[Stage 4A] 运行 5 秒 P0 冒烟自检 (校验 3072d 特征、128ch 潜变量网格与 100 步快速收敛 >= 18.5 dB)...")
    smoke_rng = random.Random(7)
    smoke_wins = []
    for i in range(2):
        f_g, x_g, t_g = engine.rollout(train_rows[i], 0.51613, True, 1000 + i)
        assert f_g.shape[-1] == 3072 and x_g.shape[0] == 128 and x_g.shape[1:] == f_g.shape[:2]
        f_s, x_s, t_s, mp = random_token_window(f_g, x_g, t_g, 24, 2, smoke_rng)
        smoke_wins.append((f_s.to(DEVICE, torch.bfloat16), x_s.to(DEVICE, torch.bfloat16), strip_px_margin(t_s.to(DEVICE, torch.float32), mp), mp))
    smoke_head = Flux3PixelHead(width=512, out_channels=3).to(DEVICE)
    smoke_opt = torch.optim.AdamW(smoke_head.parameters(), lr=8e-4, betas=(0.9, 0.99), weight_decay=0.0)
    init_p, last_p = [], []
    for st in range(1, 101):
        f_s, x_s, t_crop, mp = smoke_wins[st % 2]
        smoke_opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            pred = smoke_head(f_s, x_s)
        loss, pv = pixel_head_loss(strip_px_margin(pred, mp), t_crop)
        assert math.isfinite(float(loss.item())), "NaN/Inf Loss!"
        loss.backward()
        torch.nn.utils.clip_grad_norm_(smoke_head.parameters(), 1.0)
        smoke_opt.step()
        if st <= 4:
            init_p.append(pv)
        if st > 90:
            last_p.append(pv)
    p0_s, p0_e = sum(init_p) / len(init_p), sum(last_p) / len(last_p)
    del smoke_head, smoke_opt, smoke_wins
    torch.cuda.empty_cache()
    print(f"  P0 门禁通过！100 步 PSNR: {p0_s:.2f} dB -> {p0_e:.2f} dB")
    assert p0_e >= 18.0 and (p0_e - p0_s) >= 6.0, f"P0 爬升不足: {p0_s:.2f} -> {p0_e:.2f} dB"

head = Flux3PixelHead(width=HEAD_WIDTH, depth_low=2, depth_high=1, out_channels=3).to(DEVICE)
n_params = sum(p.numel() for p in head.parameters())
print(f"\n[Stage 4B] 正式训练 Flux3PixelHead: width={HEAD_WIDTH}, 参数量={n_params/1e6:.2f}M (~{n_params*4/(1024*1024):.1f} MiB FP32)")

total_grad_steps = TOTAL_ROLLOUTS * UPDATES_PER_ROLLOUT
optimizer = torch.optim.AdamW(head.parameters(), lr=LEARNING_RATE, weight_decay=1e-2, betas=(0.9, 0.99))


def set_step_lr(opt, cur_step: int) -> float:
    warm = min(1.0, (cur_step + 1) / max(1, WARMUP_STEPS))
    prog = min(1.0, cur_step / max(1, total_grad_steps))
    lr_now = LEARNING_RATE * warm * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * prog)))
    for pg in opt.param_groups:
        pg["lr"] = lr_now
    return lr_now


start_rollout, opt_step_count, best_eval_psnr, history = 0, 0, 0.0, []
if RESUME_IF_EXISTS and latest_ckpt.exists():
    st_dict = torch.load(latest_ckpt, map_location=DEVICE, weights_only=False)
    head.load_state_dict(st_dict["head"])
    optimizer.load_state_dict(st_dict["optimizer"])
    start_rollout = int(st_dict["rollout"])
    opt_step_count = int(st_dict.get("opt_step_count", start_rollout * UPDATES_PER_ROLLOUT))
    best_eval_psnr = float(st_dict.get("best_eval_psnr", 0.0))
    history = list(st_dict.get("history", []))
    print(f"从断点恢复: Rollout {start_rollout}/{TOTAL_ROLLOUTS} (best={best_eval_psnr:.2f} dB)")

replay = deque(maxlen=REPLAY_BUFFER_SIZE)
rng = random.Random(20261122 + start_rollout)


@torch.no_grad()
def evaluate_heldout(n_eval: int = 24) -> dict:
    head.eval()
    ps_d, ps_b = [], []
    for idx_e, row in enumerate(test_rows[: min(n_eval, len(test_rows))]):
        use_d = (idx_e % 2 == 0)
        sig = 0.51613 if use_d else 0.24615
        f_g, x_g, t_g = engine.rollout(row, sig, use_d, 880000 + idx_e)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            pred = head(f_g.unsqueeze(0).to(DEVICE, torch.bfloat16), x_g.unsqueeze(0).to(DEVICE, torch.bfloat16))
        p01 = ((pred[0].float().cpu() + 1.0) * 0.5).clamp(0, 1)
        t01 = ((t_g.float() + 1.0) * 0.5).clamp(0, 1)
        db = -10.0 * math.log10(max(1e-10, float(((p01 - t01) ** 2).mean().item())))
        (ps_d if use_d else ps_b).append(db)
    head.train()
    all_p = ps_d + ps_b
    return {"mean_psnr": sum(all_p) / len(all_p), "distilled_psnr": sum(ps_d) / max(1, len(ps_d)), "base_psnr": sum(ps_b) / max(1, len(ps_b))}


t_train = time.time()
run_loss, run_psnr, run_n = 0.0, 0.0, 0
for r_idx in range(start_rollout + 1, TOTAL_ROLLOUTS + 1):
    row = train_rows[(r_idx - 1) % len(train_rows)]
    # 每 16 步切换一次 Distilled / Base 模型以消除频繁显存切换开销
    use_d = True if (patcher_base is patcher_distilled) else (((r_idx - 1) // 16) % 2 == 0)
    if use_d:
        sigma = 0.51613 if (r_idx % 3 == 0) else rng.uniform(*DISTILLED_SIGMA_RANGE)
    else:
        sigma = rng.uniform(*BASE_SIGMA_RANGE)

    replay.append(engine.rollout(row, sigma, use_d, 100000 + r_idx))
    n_upd = UPDATES_PER_ROLLOUT if len(replay) >= 8 else 4
    cur_lr = LEARNING_RATE
    for _ in range(n_upd):
        cur_lr = set_step_lr(optimizer, opt_step_count)
        bf, bx, bt = replay[rng.randrange(len(replay))]
        f_s, x_s, t_s, mp = random_token_window(bf, bx, bt, WINDOW_TOKENS, MARGIN_TOKENS, rng)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            pred = head(f_s.to(DEVICE, torch.bfloat16), x_s.to(DEVICE, torch.bfloat16))
        loss, pv = pixel_head_loss(strip_px_margin(pred, mp), strip_px_margin(t_s.to(DEVICE, torch.float32), mp))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0)
        optimizer.step()
        opt_step_count += 1
        run_loss += float(loss.detach().item()); run_psnr += pv; run_n += 1

    if r_idx % 50 == 0 or r_idx == 1:
        el = time.time() - t_train
        eta_m = (TOTAL_ROLLOUTS - r_idx) / max(1e-5, (r_idx - start_rollout) / max(1e-3, el)) / 60.0
        print(f"[Rollout {r_idx:4d}/{TOTAL_ROLLOUTS}] loss={run_loss/run_n:.4f} | train_PSNR={run_psnr/run_n:.2f} dB | lr={cur_lr:.2e} | VRAM={torch.cuda.max_memory_allocated()/1e9:.1f}GB | ETA={eta_m:.1f}m")
        run_loss, run_psnr, run_n = 0.0, 0.0, 0

    if r_idx % SAVE_EVERY_ROLLOUTS == 0 or r_idx == TOTAL_ROLLOUTS:
        ev = evaluate_heldout(24)
        history.append({"rollout": r_idx, **ev})
        print(f"  [Eval @ {r_idx}] Mean PSNR = {ev['mean_psnr']:.2f} dB (Distilled: {ev['distilled_psnr']:.2f} dB | Base: {ev['base_psnr']:.2f} dB)")
        snap = CKPT_DRIVE / f"flux3_klein_pixel_head_r{r_idx:04d}.safetensors"
        save_flux3_pixel_head(head, snap, extra_meta={"base_model": "FLUX.3-Image-Klein", "teacher_vae": VAE_PATH.name, "rollouts": str(r_idx), "eval_psnr_db": f"{ev['mean_psnr']:.2f}"})
        if ev["mean_psnr"] >= best_eval_psnr or r_idx == TOTAL_ROLLOUTS:
            best_eval_psnr = max(best_eval_psnr, ev["mean_psnr"])
            for dst in (PIXEL_HEADS_DRIVE / "flux3_klein_pixel_head.safetensors", VAE_DRIVE / "flux3_klein_pixel_head.safetensors", CKPT_DRIVE / "flux3_klein_pixel_head_best.safetensors"):
                shutil.copyfile(snap, dst)
        torch.save({"rollout": r_idx, "opt_step_count": opt_step_count, "head": head.state_dict(), "optimizer": optimizer.state_dict(), "best_eval_psnr": best_eval_psnr, "history": history}, latest_ckpt)

print(f"\nFLUX.3-Image-Klein Pixel Head 训练完成！最优测试集 PSNR = {best_eval_psnr:.2f} dB")

In [ ]:
# @title 5. 三方基准对比评测：Official `flux2-vae` vs BFL `small_decoder` vs 你的 `Flux3PixelHead`
import math, time
from pathlib import Path
from IPython.display import Image as IPyImage, Markdown, display
from PIL import Image, ImageDraw
import lpips, numpy as np, torch
import comfy.sd, comfy.utils

lpips_vgg = lpips.LPIPS(net="vgg", verbose=False).to(DEVICE).eval()
EVAL_DRIVE = Path("/content/drive/MyDrive/FastFrames_Flux3Klein/eval_report")
EVAL_DRIVE.mkdir(parents=True, exist_ok=True)
BEST_HEAD_PATH = Path("/content/drive/MyDrive/ComfyUI_Hub/models/pixel_heads/flux3_klein_pixel_head.safetensors")

small_vae = None
small_cand = Path("/content/local_models/vae/full_encoder_small_decoder.safetensors")
if small_cand.exists():
    try:
        small_vae = comfy.sd.VAE(sd=comfy.utils.load_torch_file(str(small_cand)))
        print("已加载 BFL 官方轻量解码器 `full_encoder_small_decoder.safetensors` 同台打擂！")
    except Exception as e:
        print("⚠small_decoder 跳过:", e)

eval_head = load_flux3_pixel_head(BEST_HEAD_PATH, device=DEVICE, dtype=torch.bfloat16)
by_domain, per_ph, per_sm, lat_ph, lat_vae, lat_sm, crop_rows = {}, [], [], [], [], [], []

for idx_e, row in enumerate(test_rows[:40]):
    use_d = (idx_e % 2 == 0)
    sig = 0.51613 if use_d else 0.24615
    patcher = patcher_distilled if use_d else patcher_base
    comfy.model_management.load_models_gpu([patcher])
    bmodel = patcher.model
    x0_raw = torch.load(Path("/content/flux3_ph_cache/latents") / row["latent"], map_location=DEVICE, weights_only=True).float()
    x0_norm = bmodel.latent_format.process_in(x0_raw)
    cond = engine._get_cond(row["cond"]).to(device=DEVICE, dtype=bmodel.get_dtype_inference())
    gen = torch.Generator(device=DEVICE).manual_seed(990000 + idx_e)
    xt_norm = (1.0 - sig) * x0_norm + sig * torch.randn(x0_norm.shape, generator=gen, device=DEVICE)
    sig_t = torch.tensor([sig], device=DEVICE, dtype=torch.float32)
    topts = {"sample_sigmas": torch.tensor([sig, 0.0], device=DEVICE, dtype=torch.float32)}

    cap = {}
    def _pre_hk_eval(_m, inp):
        if "f" not in cap:
            cap["f"] = inp[0].detach()
    h_hk = bmodel.diffusion_model.final_layer["image"].linear.register_forward_pre_hook(_pre_hk_eval)
    with torch.no_grad():
        x0_hat_norm = bmodel.apply_model(xt_norm, sig_t, c_crossattn=cond, transformer_options=topts).float()
    h_hk.remove()
    x0_hat_raw = bmodel.latent_format.process_out(x0_hat_norm)

    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        ref_nhwc = teacher_vae.decode(x0_hat_raw.cpu())[..., :3].float().clamp(0, 1)
    torch.cuda.synchronize(); lat_vae.append((time.perf_counter() - t0) * 1000.0)

    f_in, x_in = cap["f"].to(DEVICE, torch.bfloat16), x0_hat_norm.to(DEVICE, torch.bfloat16)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        ph_nchw = eval_head(f_in, x_in)
    torch.cuda.synchronize(); lat_ph.append((time.perf_counter() - t0) * 1000.0)
    ph_nhwc = ((ph_nchw[:, :3].float().cpu() + 1.0) * 0.5).clamp(0, 1).movedim(1, -1)

    mse_ph = max(1e-10, float(((ph_nhwc - ref_nhwc) ** 2).mean().item()))
    psnr_ph = -10.0 * math.log10(mse_ph)
    with torch.no_grad():
        lp_ph = float(lpips_vgg(ph_nhwc.movedim(-1, 1).to(DEVICE) * 2 - 1, ref_nhwc.movedim(-1, 1).to(DEVICE) * 2 - 1).mean().item())
    by_domain.setdefault(row["domain"], []).append((psnr_ph, lp_ph))
    per_ph.append((psnr_ph, lp_ph))

    sm_nhwc = None
    if small_vae is not None:
        torch.cuda.synchronize(); t0 = time.perf_counter()
        with torch.no_grad():
            sm_nhwc = small_vae.decode(x0_hat_raw.cpu())[..., :3].float().clamp(0, 1)
        torch.cuda.synchronize(); lat_sm.append((time.perf_counter() - t0) * 1000.0)
        mse_s = max(1e-10, float(((sm_nhwc - ref_nhwc) ** 2).mean().item()))
        with torch.no_grad():
            lp_s = float(lpips_vgg(sm_nhwc.movedim(-1, 1).to(DEVICE) * 2 - 1, ref_nhwc.movedim(-1, 1).to(DEVICE) * 2 - 1).mean().item())
        per_sm.append((-10.0 * math.log10(mse_s), lp_s))

    if len(crop_rows) < 6:
        crop_rows.append((row["domain"], ref_nhwc[0], ph_nhwc[0], sm_nhwc[0] if sm_nhwc is not None else None, psnr_ph))

med_vae = sorted(lat_vae)[len(lat_vae) // 2]
med_ph = sorted(lat_ph)[len(lat_ph) // 2]
mean_psnr = sum(x[0] for x in per_ph) / len(per_ph)
mean_lp = sum(x[1] for x in per_ph) / len(per_ph)

lines = [
    "# FLUX.3-Image-Klein Direct Pixel Head — Benchmark Report",
    "",
    "| Decoder | File Size | Median Decode Latency (`768×768`) | Speedup vs VAE | PSNR (↑ dB) | LPIPS-VGG (↓) |",
    "| :--- | :---: | :---: | :---: | :---: | :---: |",
    f"| **Stock `flux2-vae` (Teacher)** | `{VAE_PATH.stat().st_size/(1024*1024):.1f} MiB` | `{med_vae:.1f} ms` | `1.0×` | *(Reference)* | `0.0000` |",
]
if per_sm:
    med_sm = sorted(lat_sm)[len(lat_sm) // 2]
    lines.append(f"| `full_encoder_small_decoder` (BFL Small VAE) | `{small_cand.stat().st_size/(1024*1024):.1f} MiB` | `{med_sm:.1f} ms` | `{med_vae/max(0.05,med_sm):.1f}×` | `{sum(x[0] for x in per_sm)/len(per_sm):.2f} dB` | `{sum(x[1] for x in per_sm)/len(per_sm):.4f}` |")
lines.append(f"| **Ours: `Flux3PixelHead`** | **`{BEST_HEAD_PATH.stat().st_size/(1024*1024):.1f} MiB`** | **`{med_ph:.2f} ms`** | **`{med_vae/max(0.05,med_ph):.1f}×`** | **`{mean_psnr:.2f} dB`** | **`{mean_lp:.4f}`** |")
lines += ["", "## Per-Domain Breakdown", "", "| Domain | Samples | PSNR (↑ dB) | LPIPS-VGG (↓) |", "| :--- | :---: | :---: | :---: |"]
for dom, vals in sorted(by_domain.items()):
    lines.append(f"| `{dom}` | {len(vals)} | `{sum(v[0] for v in vals)/len(vals):.2f} dB` | `{sum(v[1] for v in vals)/len(vals):.4f}` |")

report_md = "\n".join(lines) + "\n"
(EVAL_DRIVE / "BENCHMARK_REPORT.md").write_text(report_md, encoding="utf-8")

cols = 3 if any(c[3] is not None for c in crop_rows) else 2
cell_px = 384
grid_im = Image.new("RGB", (cols * cell_px, len(crop_rows) * cell_px), (20, 20, 24))
dr_g = ImageDraw.Draw(grid_im)
for ri, (dom, ref_t, ph_t, sm_t, p_db) in enumerate(crop_rows):
    h, w, _ = ref_t.shape
    y0, x0 = (h - cell_px) // 2, (w - cell_px) // 2
    panels = [("Stock flux2-vae", ref_t[y0:y0+cell_px, x0:x0+cell_px])]
    if cols == 3 and sm_t is not None:
        panels.append(("BFL small_decoder", sm_t[y0:y0+cell_px, x0:x0+cell_px]))
    panels.append((f"Flux3PixelHead ({p_db:.1f}dB, {med_ph:.1f}ms)", ph_t[y0:y0+cell_px, x0:x0+cell_px]))
    for ci, (lbl, patch_t) in enumerate(panels):
        pim = Image.fromarray((patch_t.numpy() * 255.0).round().astype(np.uint8))
        grid_im.paste(pim, (ci * cell_px, ri * cell_px))
        dr_g.rectangle([ci * cell_px + 6, ri * cell_px + 6, ci * cell_px + 305, ri * cell_px + 30], fill=(0, 0, 0))
        dr_g.text((ci * cell_px + 12, ri * cell_px + 10), f"[{dom}] {lbl}", fill=(255, 255, 255))

grid_path = EVAL_DRIVE / "comparison_1to1_crops.png"
grid_im.save(grid_path)
display(Markdown(report_md))
display(IPyImage(filename=str(grid_path)))

In [ ]:
# @title 6. [可选二阶 2x 分支] 在 1x 头基础上加训 `2x` 超分直出头（1024 潜空间输出 2048）
TRAIN_X2_STAGE = False  # @param {type:"boolean"}
X2_ROLLOUTS = 1500      # @param {type:"integer"}

if not TRAIN_X2_STAGE:
    print("默认跳过；需要产出 `flux3_klein_pixel_head_x2.safetensors` 时勾选后运行本格即可。")
else:
    import random, shutil, time, torch, torch.nn.functional as F
    from pathlib import Path
    Flux3PixelHeadX2 = ph_mod.Flux3PixelHeadX2
    base_1x = load_flux3_pixel_head("/content/drive/MyDrive/ComfyUI_Hub/models/pixel_heads/flux3_klein_pixel_head.safetensors", device=DEVICE, dtype=torch.float32)
    for p in base_1x.parameters():
        p.requires_grad_(False)
    head_x2 = Flux3PixelHeadX2(base=base_1x, depth_x2=2, ls_init=1.0).to(DEVICE)
    opt_x2 = torch.optim.AdamW([p for p in head_x2.parameters() if p.requires_grad], lr=2e-4, weight_decay=1e-4)
    rng2 = random.Random(2026)
    t_x2 = time.time()
    for r_i in range(1, X2_ROLLOUTS + 1):
        row = train_rows[(r_i * 3571) % len(train_rows)]
        f_g, x_g, t_g = engine.rollout(row, 0.51613 if r_i % 2 else rng2.uniform(*BASE_SIGMA_RANGE), bool(r_i % 2), 500000 + r_i)
        t_2x = F.interpolate(t_g.unsqueeze(0).to(DEVICE, torch.float32), scale_factor=2.0, mode="bicubic", align_corners=False).clamp(-1, 1)
        gh, gw = f_g.shape[:2]
        span = min(gh, gw, 24)
        y0, x0 = rng2.randint(0, gh - span), rng2.randint(0, gw - span)
        opt_x2.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            pred_2x = head_x2(f_g[y0:y0+span, x0:x0+span].unsqueeze(0).to(DEVICE, torch.bfloat16), x_g[:, y0:y0+span, x0:x0+span].unsqueeze(0).to(DEVICE, torch.bfloat16), scale=2)
        loss_2x, psnr_2x = pixel_head_loss(pred_2x[:, :, 32:-32, 32:-32], t_2x[:, :, y0*32+32:(y0+span)*32-32, x0*32+32:(x0+span)*32-32])
        loss_2x.backward(); torch.nn.utils.clip_grad_norm_(head_x2.parameters(), 1.0); opt_x2.step()
        if r_i % 100 == 0:
            print(f"[2x Rollout {r_i}/{X2_ROLLOUTS}] 2x_PSNR={psnr_2x:.2f} dB | {time.time()-t_x2:.0f}s")
    out_x2 = Path("/content/drive/MyDrive/ComfyUI_Hub/models/pixel_heads/flux3_klein_pixel_head_x2.safetensors")
    save_flux3_pixel_head(head_x2, out_x2, extra_meta={"scale": "2", "rollouts_x2": str(X2_ROLLOUTS)})
    shutil.copyfile(out_x2, Path("/content/drive/MyDrive/ComfyUI_Hub/models/vae/flux3_klein_pixel_head_x2.safetensors"))
    print("已保存:", out_x2)